# 03 — PDE Finite Difference Methods

This notebook explores the PDE solver configuration knobs:

1. **Crank-Nicolson** with Rannacher smoothing (default)
2. **Explicit** scheme — and the `StabilityError` it can produce
3. **Explicit with log-spot grid** — stable explicit stepping
4. **Intrinsic vs Gauss-Seidel** for early exercise
5. **Implicit** scheme

## 1) Setup

In [1]:
import datetime as dt

import derivatives_pricing as dp

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)
r = 0.05

curve = dp.DiscountCurve.flat(rate=r, end_time=2.0)
market = dp.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")
underlying = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market)

eu_put = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

am_put = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.AMERICAN,
    strike=100.0,
    maturity=maturity,
)

## 2) Crank-Nicolson with Rannacher Smoothing (Default)

The default PDE configuration uses Crank-Nicolson time-stepping with 2
Rannacher smoothing steps at inception. This is the recommended starting
point.

In [3]:
pde_cn = dp.OptionValuation(
    underlying=underlying,
    spec=eu_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(),  # defaults: method=CRANK_NICOLSON, rannacher_steps=2
)
print(f"Crank-Nicolson (default): {pde_cn.present_value():.6f}")

# Compare to BSM
bsm = dp.OptionValuation(underlying=underlying, spec=eu_put, pricing_method=dp.PricingMethod.BSM)
print(f"BSM analytical:           {bsm.present_value():.6f}")

Crank-Nicolson (default): 5.563611
BSM analytical:           5.573526


## 3) Explicit Scheme — Stability Error

The explicit scheme has a CFL stability condition. With too few time steps
relative to spot steps, the solver raises a `StabilityError`.

In [4]:
try:
    pde_explicit = dp.OptionValuation(
        underlying=underlying,
        spec=eu_put,
        pricing_method=dp.PricingMethod.PDE_FD,
        params=dp.PDEParams(
            method=dp.PDEMethod.EXPLICIT,
            spot_steps=200,
            time_steps=200,  # not enough for explicit stability
        ),
    )
    pde_explicit.present_value()
except dp.StabilityError as e:
    print(f"StabilityError: {e}")

StabilityError: Explicit spot-grid scheme likely unstable (CFL violation, pure explicit discounting). max_dt=0.005 exceeds dt_max=0.000625. Increase time_steps to >= 1601, or use log-spot/implicit/CN.


## 4) Explicit with Log-Spot Grid

Switching to a **log-spot** grid changes the PDE coefficients so that the
explicit scheme becomes stable for a wider range of parameters.

In [5]:
pde_explicit_log = dp.OptionValuation(
    underlying=underlying,
    spec=eu_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(
        method=dp.PDEMethod.EXPLICIT,
        space_grid=dp.PDESpaceGrid.LOG_SPOT,
        spot_steps=200,
        time_steps=200,
    ),
)
print(f"Explicit + log grid: {pde_explicit_log.present_value():.6f}")
print(f"BSM:                 {bsm.present_value():.6f}")

Explicit + log grid: 5.564244
BSM:                 5.573526


## 5) Hull's Explicit Scheme

`EXPLICIT_HULL` is an alternative explicit formulation from Hull's textbook.

In [6]:
pde_hull = dp.OptionValuation(
    underlying=underlying,
    spec=eu_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(
        method=dp.PDEMethod.EXPLICIT_HULL,
        space_grid=dp.PDESpaceGrid.LOG_SPOT,
        spot_steps=200,
        time_steps=1000,
    ),
)
print(f"Explicit Hull: {pde_hull.present_value():.6f}")
print(f"BSM:           {bsm.present_value():.6f}")

Explicit Hull: 5.569911
BSM:           5.573526


## 6) Implicit Scheme

The implicit scheme is unconditionally stable but only first-order accurate
in time.

In [7]:
pde_impl = dp.OptionValuation(
    underlying=underlying,
    spec=eu_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(method=dp.PDEMethod.IMPLICIT),
)
print(f"Implicit:      {pde_impl.present_value():.6f}")
print(f"BSM:           {bsm.present_value():.6f}")

Implicit:      5.558959
BSM:           5.573526


## 7) Early Exercise — Intrinsic vs Gauss-Seidel

For American options, the PDE solver must enforce the early-exercise
constraint. Two approaches are available:

- **`INTRINSIC`** — simple projection: $V = \max(V, \text{payoff})$ after each step
- **`GAUSS_SEIDEL`** — projected SOR iteration (more accurate, especially for
  implicit/Crank-Nicolson schemes)

In [8]:
pde_am_gs = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(american_solver=dp.PDEEarlyExercise.GAUSS_SEIDEL),
)

pde_am_intr = dp.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(american_solver=dp.PDEEarlyExercise.INTRINSIC),
)

print(f"Gauss-Seidel: {pde_am_gs.present_value():.6f}")
print(f"Intrinsic:    {pde_am_intr.present_value():.6f}")
print(f"Difference:   {pde_am_gs.present_value() - pde_am_intr.present_value():.6f}")

Gauss-Seidel: 6.078784
Intrinsic:    6.076248
Difference:   0.002536


## 8) Method Comparison

In [9]:
print(f"{'Method':<35} {'European Put':>13}")
print("-" * 50)
for label, val in [
    ("BSM (analytical)", bsm),
    ("Crank-Nicolson + Rannacher", pde_cn),
    ("Implicit", pde_impl),
    ("Explicit (log grid)", pde_explicit_log),
    ("Explicit Hull", pde_hull),
]:
    print(f"{label:<35} {val.present_value():>13.6f}")

Method                               European Put
--------------------------------------------------
BSM (analytical)                         5.573526
Crank-Nicolson + Rannacher               5.563611
Implicit                                 5.558959
Explicit (log grid)                      5.564244
Explicit Hull                            5.569911
